In [ ]:
# 1. Dọn dẹp sạch sẽ các phiên bản cũ
!pip uninstall -y transformers huggingface_hub peft bitsandbytes

# 2. Cài đặt các thư viện nền tảng (ẩn log cho gọn)
!pip install -q torch torchvision torchaudio accelerate tqdm

# 3. Cài đặt thư viện lõi
!pip install --upgrade transformers huggingface_hub peft

# 4. Cài đặt CUDA và bitsandbytes
!pip install -q nvidia-nvjitlink-cu12
!pip install -q --upgrade bitsandbytes

In [ ]:
# ==========================================
# 1. IMPORT THƯ VIỆN (Bắt buộc chạy lại sau khi Restart Runtime)
# ==========================================
import json
import os
import re
import gc
import time
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ==========================================
# 2. CẤU HÌNH VÀ TẢI MODEL
# ==========================================
INPUT_PATH = "/kaggle/input/datasets/nhimchauphii/nhimhoang/test_part1_200.json"   
OUTPUT_PATH = "t2_qwen35_9b_outputs_part1.json" 

BASE_MODEL_ID = "techwithsergiu/Qwen3.5-text-9B-bnb-4bit" 
LORA_MODEL_ID = "Nhat-Quang/outfitmatch-stylist-final-qwen35-9b-bnb4-lora"

print("🔄 Đang cấu hình và tải Base Model...")

# Tải Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# Tải Base Model (mô hình đã được BNB 4-bit)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print(f"Đang đắp LoRA adapter ({LORA_MODEL_ID}) lên Base Model...")
model = PeftModel.from_pretrained(base_model, LORA_MODEL_ID)
model.eval()
print("Tải Model thành công!\n")


In [ ]:
# ==========================================
# 2. HÀM TÌM KIẾM TÀI LIỆU (RETRIEVAL)
# ==========================================
def retrieve_documents(item):
    """
    Nhiệm vụ: Lấy trực tiếp tập ngữ cảnh chuẩn đã được gán nhãn sẵn trong file test.
    """
    return item.get("contexts", [])

In [ ]:
# ==========================================
# 3. HÀM SINH CÂU TRẢ LỜI (Ô SỐ 6)
# ==========================================
def generate_answer(question: str, retrieved_contexts: list) -> str:
    if isinstance(retrieved_contexts, list):
        context_text = "\n- ".join([str(c) for c in retrieved_contexts])
    else:
        context_text = str(retrieved_contexts)

    if not context_text.strip():
        return "Tôi không tìm thấy thông tin."

    # Định dạng tin nhắn chuẩn của Qwen (chỉ chứa text thuần túy)
    messages = [
        {
            "role": "system", 
            "content": "Bạn là chuyên gia tư vấn thời trang. Hãy trả lời câu hỏi CHỈ DỰA TRÊN tài liệu tham khảo. Nếu tài liệu không chứa thông tin, hãy nói chính xác: 'Tôi không tìm thấy thông tin'."
        },
        {
            "role": "user", 
            "content": f"--- TÀI LIỆU THAM KHẢO ---\n{context_text}\n--------------------------\nCâu hỏi: {question}"
        }
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    
    # Tính toán chiều dài đầu vào để loại bỏ prompt lúc in kết quả
    input_length = inputs["input_ids"].shape[1]
    
    # Lấy ID của thẻ đóng chat <|im_end|> để ép model dừng đúng lúc
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    eos_ids = [tokenizer.eos_token_id]
    if im_end_id is not None and not isinstance(im_end_id, list):
        eos_ids.append(im_end_id)
    elif isinstance(im_end_id, list):
        eos_ids.extend(im_end_id)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=512, 
            temperature=0.1,    
            do_sample=True,
            top_p=0.9,
            use_cache=True,
            eos_token_id=eos_ids
        )
        
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    # Qwen3.5-9B-Instruct không phải là model suy nghĩ (Thinking), nên không cần xử lý thẻ <think>
    return response.strip()


In [ ]:
# ==========================================
# 4. VÒNG LẶP CHÍNH (Ô SỐ 7)
# ==========================================
def run_evaluation_pipeline(input_file: str, output_file: str, save_interval: int = 10, limit: int = None):
    dataset = []
    
    # 1. Đọc dữ liệu (Checkpoint hoặc Khới tạo mới)
    if os.path.exists(output_file):
        print(f"🔄 Đang tải Checkpoint từ: {output_file}...")
        with open(output_file, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
    else:
        print(f"📖 Đang đọc file gốc: {input_file}...")
        with open(input_file, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
            
    if limit is not None:
        print(f"⚠️ Giới hạn xử lý {limit} câu đầu tiên để test.")
        dataset = dataset[:limit]
            
    uncompleted_items = [item for item in dataset if "answer" not in item]
    completed_samples = len(dataset) - len(uncompleted_items)
    
    print(f"🚀 Tiến trình: {completed_samples}/{len(dataset)} câu đã hoàn thành.")
    
    # 2. Chạy vòng lặp
    for idx, item in enumerate(tqdm(
        uncompleted_items, 
        desc="Đang sinh câu trả lời", 
        initial=completed_samples, 
        total=len(dataset)
    )):
        question = item.get("question", "")
        
        # Bốc context có sẵn
        retrieved = retrieve_documents(item)
        item["retrieved_contexts"] = retrieved
        
        # Gọi model
        try:
            item["answer"] = generate_answer(question, retrieved)
        except Exception as e:
            item["answer"] = f"Lỗi sinh text: {str(e)}"
        
        # 3. CHỈ LƯU CHECKPOINT SAU MỖI 10 CÂU (Bảo vệ ổ cứng Kaggle)
        if (idx + 1) % save_interval == 0 or (idx + 1) == len(uncompleted_items):
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(dataset, f, ensure_ascii=False, indent=2)
                
        # Dọn rác bộ nhớ sau mỗi câu hỏi để tránh lỗi Out Of Memory (OOM) GPU
        gc.collect()
        torch.cuda.empty_cache()
                
    print(f"\n🎉 Hoàn thành! Kết quả lưu tại: {output_file}")

In [ ]:
# Gọi hàm để bắt đầu chạy vòng lặp 200 câu
run_evaluation_pipeline(INPUT_PATH, OUTPUT_PATH, limit=5)